# 10. Trotter cochain extinction and finite spectral invariants

**Level:** Advanced

**Prerequisites:** Familiarity with finite Fock space CAR operators and exact rational matrices.

**Canonical claim anchors:** `WDP-GATE-E`, `WDP-GATE-G`, `WDP-TROTTER-01`

## Learning objectives

- construct the exact rational response matrix $G^{(2)}_{4,M}$ for Trotter steps $M \in \{1, 2, 3, 4\}$
- check the zero-mode count at the stated coupling and four values of M $\dim \ker G = 4M - 1$
- evaluate the symbolic closed rational forms for $\lambda_{\mathrm{stag}}(r, M)$ at $M=1$ and $M=2$
- distinguish coordinate Hessian instability from a rescaled quadratic response and its separate asymptotic assumptions

Work top-to-bottom. Predict each worked result before executing it, then use
the exercise to test whether you can separate the finite result from its scope boundary.


In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
root = next((candidate for candidate in (start, *start.parents)
             if (candidate / 'pyproject.toml').is_file()
             and (candidate / 'code' / 'witnessed_difference').is_dir()), None)
if root is None:
    candidate = start / 'atlas/papers/finite_obstruction_calculus/witnessed_difference_programme'
    if (candidate / 'pyproject.toml').is_file() and (candidate / 'code' / 'witnessed_difference').is_dir():
        root = candidate
if root is None:
    raise RuntimeError('could not locate the self-contained paper package')
sys.path.insert(0, str(root / 'code'))


## Idea: Exact Rational Trotter Response

In the $L=4$ staggered fermion transfer-matrix formulation, each time step is decomposed
into $2M$ alternating transfer layers $T_0$ (even spatial bonds) and $T_1$ (odd spatial bonds).
The response matrix $G^{(2)}_{4,M}$ records the exact second derivatives of the partition
trace with respect to local source fields $z_{j,x}$.

Here we import `trotter_characteristic_polynomials` to inspect the exact rational
closed forms and spectral decomposition.


In [ ]:
import sympy as sp
from trotter_characteristic_polynomials import (
    closed_form_lambda_stag_m1,
    closed_form_lambda_stag_m2,
    compute_exact_response_matrix,
)

r = sp.Rational(1, 10)
val_m1 = closed_form_lambda_stag_m1(r)
val_m2 = closed_form_lambda_stag_m2(r)

print(f"lambda_stag(1/10, M=1) = {val_m1} (~ {float(val_m1):.6e})")
print(f"lambda_stag(1/10, M=2) = {val_m2} (~ {float(val_m2):.6e})")


## Visual Checkpoint: Zero-Mode Dimension and Negative Eigenvalues

We compute $G^{(2)}_{4,M}$ for $M \in \{1, 2, 3, 4\}$ in exact rational arithmetic.
We check:
1. Total dimension: $8M$
2. Nullity (zero modes): exactly $4M - 1$ ($3, 7, 11, 15$)
3. Negative eigenvalues: strictly $1$ mode, matching $\lambda_{\mathrm{stag}}$


In [ ]:
table = []
for M in (1, 2, 3, 4):
    res = compute_exact_response_matrix(M, r)
    table.append({
        "M": M,
        "dim_G": res["dim_G"],
        "zero_modes": res["zero_modes"],
        "expected_zeros": 4 * M - 1,
        "zero_law_holds": res["zero_modes"] == 4 * M - 1,
        "negative_modes": res["negative_count"],
        "lambda_stag": res["rayleigh_stag"],
        "lambda_stag_float": f"{float(sp.Rational(res['rayleigh_stag'])):.6e}",
    })

import pprint
pprint.pprint(table)


## Rescaled quadratic response

For the staggered vector $v$ used here, $\lVert v\rVert^2=8M$. Its rescaled
quadratic response is $v^T G v/(4M^2)=2\lambda_{\mathrm{stag}}/M$.
It is negative in these fixtures, so it is not the variance of a real random
variable under a positive probability measure. Four finite values do not
establish an asymptotic rate. A bound on $\lambda_{\mathrm{stag}}$ uniform
in $M$ is required before inferring one.


In [ ]:
print("Rescaled quadratic response at the four checked M values:")
for entry in table:
    M = entry["M"]
    lam = sp.Rational(entry["lambda_stag"])
    response = sp.Rational(entry["dim_G"], 4 * M**2) * lam
    print(f"M={M}: response={response} (~ {float(response):.6e})")


## Exercise

Evaluate the ratio between $\lambda_{\mathrm{stag}}(r, M=2)$ and $\lambda_{\mathrm{stag}}(r, M=1)$ at leading
order as $r \to 0$. What is the ratio $\lim_{r \to 0} \lambda_{\mathrm{stag}}(r, M=2) / \lambda_{\mathrm{stag}}(r, M=1)$?


In [ ]:
r_sym = sp.Symbol('r')
m1_lead = sp.series(closed_form_lambda_stag_m1(r_sym), r_sym, 0, 6)
m2_lead = sp.series(closed_form_lambda_stag_m2(r_sym), r_sym, 0, 6)
print("M=1 leading term:", m1_lead)
print("M=2 leading term:", m2_lead)
ratio = sp.limit(closed_form_lambda_stag_m2(r_sym) / closed_form_lambda_stag_m1(r_sym), r_sym, 0)
print("Ratio M=2 / M=1 as r -> 0:", ratio)
print(f"This 1/4 ratio concerns M=1 and M=2 only; it is not an all-M scaling theorem.")


## Evidence scope

At $r=1/10$, exact computation checks nullity $4M-1$ and one negative
eigenvalue for $M=1,2,3,4$. It supplies neither an all-M spectral theorem nor
a continuum variance. The M=1 closed formula has expansion
$-r^4/64+r^5/64+O(r^6)$, so the formerly stated $O(r^6)$ remainder after
only the fourth-order term was false. The M=2/M=1 leading coefficient ratio
is $1/4$. Larger spatial carriers and uniform asymptotic bounds remain separate
questions.


## Continue the learning path

- Previous: [Mutation recovery, evidence composition, and terminal scope](../090_mutation_recovery_and_scope.ipynb)
- Index: [Notebook curriculum](../README.md)
- Companion paper chapter: [Trotter cochain extinction derivation](../../docs/meron_field_limit_derivations/trotter_cochain_extinction_derivation.md)
